# Build and upload index

## Shared variables

In [1]:
import pathlib
import os

# `APUZ_PART` for testing/subsets and `APUZ` for the complete index
# sub_dir = '/APUZ_PART_POPULISMUS'
sub_dir = '/APUZ'
document_input_dir = f"..{sub_dir}/corpus"
# List all pdf files (we do not need to index the metadata files)
pdf_files = [
    str(p) for p in pathlib.Path(document_input_dir).rglob("*") 
    if p.is_file() and (p.name.endswith(".pdf") or p.name.endswith(".PDF"))
]
metadata_dir = document_input_dir or os.path.dirname(pdf_files[0])
# metadata_dir = "../my-evse/knowledge_base/data_files"
print(f"Metadata directory: {metadata_dir}")

window_size = 3


Metadata directory: ../APUZ/corpus


## Load meta data from yaml files

In [2]:
from loguru import logger
import pathlib
import yaml
from llama_index.core import SimpleDirectoryReader
from pprint import pprint
from typing import Dict
import os
import json
#import chardet 

metadata_dict = {}

# load and parse all yaml files in metadata_dir
for filepath in pathlib.Path(metadata_dir).rglob("*.yaml"):
    logger.info(f"Loading metadata from {filepath}")

    # if the corresponding pdf file does not exist, skip this metadata file
    pdf_file = os.path.join(metadata_dir, filepath.stem + ".pdf")
    if not pathlib.Path(pdf_file).is_file():
        logger.warning(f"PDF file {pdf_file} does not exist, skipping metadata file {filepath}.")
        continue
    with open(filepath, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
        if (
            "file" in data 
            and "author" in data 
            and "url" in data
            and "title" in data
        ):
            filename = data["file"]
            metadata_dict[filename] = {
                "author": data["author"],
                "title": data["title"], 
                "url": data["url"],
                "year": data.get("year", None),  # optional field
                "month": data.get("month", None),  # optional field
                # added by hand (not correctly in the yaml files/bibtex file)
                "journal": "Aus Politik und Zeitgeschichte (APuZ)",
            }
        else:
            logger.warning(f"Invalid metadata in {filepath}.")

# serializing the meta_dict as json file
# can be use to pass it to the builder (alternativley, we pass a function)
meta_json_path = os.path.join(metadata_dir, "metadata_dict.json")
with open(meta_json_path, "w", encoding="utf-8") as json_file:
    json.dump(metadata_dict, json_file, ensure_ascii=False, indent=2)
logger.info(f"Serialized metadata_dict to {meta_json_path}")



def metadata_func(filename: str) -> Dict: 
    meta = metadata_dict.get(pathlib.Path(filename).name, {})
    if not meta:
        logger.warning(f"No metadata found for file: {filename}")
    return meta



2025-10-04 22:39:59.143 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/veith_demokratische_2024.yaml
2025-10-04 22:39:59.150 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/bucking_kleine_2024.yaml
2025-10-04 22:39:59.153 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/klimeniouk_gibt_2024.yaml
2025-10-04 22:39:59.154 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/murciano_7_2024.yaml
2025-10-04 22:39:59.155 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/becker_antisemitische_2024.yaml
2025-10-04 22:39:59.156 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/c_politische_2024.yaml
2025-10-04 22:39:59.157 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/schlosmacher_bonn_2024.yaml
2025-10-04 22:39:59.157 | INFO     | __main__:<module>:15 - Loading metadata from ../APUZ/corpus/bonkost_erscheinung_2024.yaml
2025-10-0

### Debugging Metadata Loading Issues


In [3]:
# for debugging purposes: 
print(f"Input files: {len(pdf_files)}")
reader = SimpleDirectoryReader(
    input_files=pdf_files,
    file_metadata=metadata_func,
)
docs = reader.load_data()
print(f"Loaded {len(docs)} documents with metadata from {metadata_dir}")
mean_size = sum(len(doc.text) for doc in docs) / len(docs) if docs else 0
print(f"Mean document size (in characters): {mean_size:.2f}")
file_names = set([doc.metadata.get("file_name") for doc in docs])
print(f"Files in the index: {file_names}")


from llama_index.core.node_parser import SentenceWindowNodeParser

logger.debug("Parsing nodes...")
nodes = SentenceWindowNodeParser.from_defaults(
    window_size=window_size,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
).get_nodes_from_documents(docs, show_progress=True)

print(f"Parsed {len(nodes)} nodes from {len(docs)} documents.")
file_names2 = set([node.metadata.get("file_name") for node in nodes])
print(f"Files from nodes: {file_names2}")
print(
    f"For the following files, we could not construe nodes: {file_names - file_names2}.\n"
    "There might be an issue with the corresponding files."
)


Input files: 170


2025-10-04 22:40:54.002 | DEBUG    | __main__:<module>:17 - Parsing nodes...


Loaded 898 documents with metadata from ../APUZ/corpus
Mean document size (in characters): 5381.35
Files in the index: {'stephan_europa_2024.pdf', 'aliev_zwischen_2024.pdf', 'bucking_kleine_2024.pdf', 'romaniec_vom_2024.pdf', 'broermann_wertvolle_2024.pdf', 'stefan_preis_2024.pdf', 'wurdemann_israel_2024.pdf', 'herbert_ziemlich_2024.pdf', 'erlach_sind_2023.pdf', 'yelizaveta_teile_2024.pdf', 'reemtsma_antisemitismus_2024.pdf', 'trenz_politik_2024.pdf', 'scheller_vom_2024.pdf', 'grosmann_gefahrliche_2024.pdf', 'jana_moldau_2024.pdf', 'becker_antisemitische_2024.pdf', 'thranhardt_schrittweise_2024.pdf', 'grzegorz_gesellschaftsdienst_2024.pdf', 'diehl_rechtspopulismus_2024.pdf', 'bayramoglu_white_2024.pdf', 'genner_blackout_2023.pdf', 'ruge_souveraner_2024.pdf', 'rieger-ladich_neustart_2024.pdf', 'bretschneider_trotzdem_2024.pdf', 'matzing_von_2024.pdf', 'karoline_nachhaltig_2024.pdf', 'van_dyck_privateigentum_2024.pdf', 'irme_pladoyer_2024.pdf', 'dieter_wer_2024.pdf', 'melanie_von_2024.pd

Parsing nodes:   0%|          | 0/898 [00:00<?, ?it/s]

Parsed 38066 nodes from 898 documents.
Files from nodes: {'stephan_europa_2024.pdf', 'aliev_zwischen_2024.pdf', 'bucking_kleine_2024.pdf', 'romaniec_vom_2024.pdf', 'broermann_wertvolle_2024.pdf', 'stefan_preis_2024.pdf', 'wurdemann_israel_2024.pdf', 'herbert_ziemlich_2024.pdf', 'erlach_sind_2023.pdf', 'yelizaveta_teile_2024.pdf', 'reemtsma_antisemitismus_2024.pdf', 'trenz_politik_2024.pdf', 'scheller_vom_2024.pdf', 'grosmann_gefahrliche_2024.pdf', 'jana_moldau_2024.pdf', 'becker_antisemitische_2024.pdf', 'thranhardt_schrittweise_2024.pdf', 'grzegorz_gesellschaftsdienst_2024.pdf', 'diehl_rechtspopulismus_2024.pdf', 'bayramoglu_white_2024.pdf', 'genner_blackout_2023.pdf', 'ruge_souveraner_2024.pdf', 'rieger-ladich_neustart_2024.pdf', 'bretschneider_trotzdem_2024.pdf', 'matzing_von_2024.pdf', 'karoline_nachhaltig_2024.pdf', 'van_dyck_privateigentum_2024.pdf', 'irme_pladoyer_2024.pdf', 'dieter_wer_2024.pdf', 'melanie_von_2024.pdf', 'm_gut_2024.pdf', 'marie-luise_anatomie_2024.pdf', 'kinder

## Init configs

In [ ]:
from evidence_seeker import RetrievalConfig

embed_model_confs = [
    # ('sentence-transformers/paraphrase-multilingual-mpnet-base-v2', 'huggingface', False),
    ('BAAI/bge-m3', 'huggingface', False),
    # Not accessible via inference provider & 
    # not executable locally (on my [BC] machine)
    # 'Snowflake/snowflake-arctic-embed-m-v2.0',
    # ('sentence-transformers/all-MiniLM-L6-v2', 'huggingface', False),
    # ('jinaai/jina-embeddings-v2-base-de', 'huggingface', False),
    # ('mixedbread-ai/deepset-mxbai-embed-de-large-v1', 'huggingface_instruct_prefix', True),
]

configs = [
    RetrievalConfig(
        ###### MODEL CONFIGURATION ##############################
        env_file="../.env",
        ### Using local embedding model (via Huggingface API) ###
        # embed_backend_type="huggingface",
        # embed_model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        ### Huggingface inference API ###########
        # embed_backend_type="huggingface_inference_api",
        # # embed_model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        # embed_model_name=embed_model_name,
        # embed_base_url="https://router.huggingface.co/hf-inference/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        # api_key_name="hf_debatelab_inference_provider",
        # bill_to="DebateLabKIT",
        ### Local Model via Huggingface ###########
        embed_backend_type=backend_type,
        # instruct prefix for mxbai
        # embed_backend_type="huggingface_instruct_prefix",
        embed_model_name=embed_model_name,
        # used for snowflake-arctic-embed-m-v2.0 and mxbai
        trust_remote_code=trust_remote_code,
        ####### END MODEL CONFIGURATIAN #########
        # hub_key_name="hf_evse_data",
        document_input_dir=document_input_dir,
        document_input_files=pdf_files,
        index_persist_path=f"..{sub_dir}/storage/{embed_model_name.split('/', 1)[1]}",
        window_size=window_size,
        # uncomment the following line to upload the index to the HF hub
        #index_hub_path = "DebateLabKIT/apuz-index-es"
    ) 
    for embed_model_name, backend_type, trust_remote_code in embed_model_confs
]


2025-10-05 13:01:01.679 | WARNING  | evidence_seeker.retrieval.config:check_hub_token_name:137 - Check whether you need a HF hub token for saving/loading your index to/from the Hugging Face Hub. If you need one, set an `hub_key_name` in the retriever config and provide the token as an environment variable with that name.
2025-10-05 13:01:01.680 | WARNING  | evidence_seeker.retrieval.config:load_env_file:203 - Environment file '../.env' does not exist. Please provide a valid path to the environment file. Or set it to None if you don't need it and set the API keys in other ways as environment variables.
2025-10-05 13:01:01.680 | INFO     | evidence_seeker.retrieval.config:load_env_file:208 - Loaded environment variables from '../.env'


In [ ]:
for embed_model_name, backend_type, trust_remote_code in embed_model_confs:
    print(embed_model_name.split('/', 1)[1])


deepset-mxbai-embed-de-large-v1


## Build index

In [9]:
from evidence_seeker.retrieval.index_builder import IndexBuilder

for config in configs:
    index_builder = IndexBuilder(config=config)
    index_builder.build_index(metadata_func=metadata_func)

2025-10-05 13:01:06.631 | WARNING  | evidence_seeker.retrieval.index_builder:__init__:53 - No API token provided for embedding model.
2025-10-05 13:01:06.631 | DEBUG    | evidence_seeker.retrieval.base:_get_embed_model:236 - Inititializing embed model: mixedbread-ai/deepset-mxbai-embed-de-large-v1 
2025-10-05 13:01:06,632 - INFO - Load pretrained SentenceTransformer: mixedbread-ai/deepset-mxbai-embed-de-large-v1
2025-10-05 13:01:09,570 - INFO - 2 prompts are loaded, with the keys: ['query', 'passage']
2025-10-05 13:01:09.570 | WARNING  | evidence_seeker.retrieval.index_builder:_get_callback_manager:104 - If you want to track progress, you have to pass a progress callback on init.
2025-10-05 13:01:09.571 | WARNING  | evidence_seeker.retrieval.index_builder:_load_documents:759 - Both 'document_input_dir' and 'document_input_files'  provided'. Using 'document_input_files'.
2025-10-05 13:01:09.571 | DEBUG    | evidence_seeker.retrieval.index_builder:_load_documents:768 - Reading documents 

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

...

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1202 [00:00<?, ?it/s]

2025-10-05 18:32:35.625 | DEBUG    | evidence_seeker.retrieval.index_builder:_post_parsing_consistency_checks:848 - Number of parsed nodes the index: 38066
2025-10-05 18:32:36.523 | DEBUG    | evidence_seeker.retrieval.index_builder:_persist_index:874 - Persisting index to /home/basti/Nextcloud/Documents/mindmaps/mind/projects/kideku/code/evidence-seeker/TMP/APUZ/storage/deepset-mxbai-embed-de-large-v1/index


### Upload index to Hugging Face Hub

In [ ]:
# explicitly upload the index to the HuggingFace hub 
# (can also be done via the `build_index` function)
from evidence_seeker.retrieval.base import INDEX_PATH_IN_REPO
import huggingface_hub

# Load the HuggingFace API token from the environment variable
hub_token = os.getenv(config.hub_key_name, None)

HfApi = huggingface_hub.HfApi(token=hub_token)
HfApi.upload_folder(
    repo_id="DebateLabKIT/apuz-index-es",
    folder_path=config.index_persist_path,
    #path_in_repo=INDEX_PATH_IN_REPO,
    repo_type="dataset",
)

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

default__vector_store.json:   0%|          | 0.00/650M [00:00<?, ?B/s]

docstore.json:   0%|          | 0.00/208M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/DebateLabKIT/apuz-index-es/commit/799dee54e009b532410c4765368ee1bcbdade16b', commit_message='Upload folder using huggingface_hub', commit_description='', oid='799dee54e009b532410c4765368ee1bcbdade16b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/DebateLabKIT/apuz-index-es', endpoint='https://huggingface.co', repo_type='dataset', repo_id='DebateLabKIT/apuz-index-es'), pr_revision=None, pr_num=None)

In [3]:
# explicitly upload the index to the HuggingFace hub 
# (can also be done via the `build_index` function)
from evidence_seeker.retrieval.base import INDEX_PATH_IN_REPO
import huggingface_hub

hub_key_name="hf_evse_data"
index_persist_path="/home/basti/tmp/APUZ/storage/bge-m3"

# Load the HuggingFace API token from the environment variable
hub_token = os.getenv(hub_key_name, None)

HfApi = huggingface_hub.HfApi(token=hub_token)
HfApi.upload_folder(
    repo_id="DebateLabKIT/apuz-index-bge-m3",
    folder_path=index_persist_path,
    #path_in_repo=INDEX_PATH_IN_REPO,
    repo_type="dataset",
)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  .../storage/bge-m3/index/docstore.json:   2%|1         | 4.38MB /  229MB            

  ...m3/index/default__vector_store.json:   0%|          |  658kB /  936MB            

  2025-10-13T11:48:49.364983Z  WARN  Status Code: 502. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:220

  2025-10-13T11:49:30.781077Z  WARN  Status Code: 502. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:220



CommitInfo(commit_url='https://huggingface.co/datasets/DebateLabKIT/apuz-index-bge-m3/commit/18bfea73ebc4d87b025af126fef35699b6ea01bb', commit_message='Upload folder using huggingface_hub', commit_description='', oid='18bfea73ebc4d87b025af126fef35699b6ea01bb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/DebateLabKIT/apuz-index-bge-m3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='DebateLabKIT/apuz-index-bge-m3'), pr_revision=None, pr_num=None)